# 99 - Teardown: delete everything SafeDrive created

Run this when you have **finished all practice**. Each notebook already cleans up its own
short-lived resources; this notebook removes what was kept for later notebooks, and sweeps for
anything named `safedrive*` that was left behind.

- Every cell lists what it finds and then deletes it. Errors are printed, not raised.
- Nothing is deleted until you set `CONFIRM = True`.
- The S3 bucket and the IAM role go last (sections 9 and 10).
- Not removable: CloudWatch metrics and finished job records (training, processing, evaluation). They cost nothing.

In [ ]:
import sys; sys.path.append("..")
from common import utils as U, catchup as C
print(U.REGION, U.account_id())

**Safety switch.** Set to `True` to really delete.

In [ ]:
CONFIRM = False

Two small helpers: `mine()` filters names that belong to this project, `drop()` deletes when confirmed.

In [ ]:
def mine(name):
    return str(name).lower().startswith("safedrive")

def drop(fn, **kwargs):
    label = ", ".join(str(v) for v in kwargs.values())[:90]
    if not CONFIRM:
        print("would delete:", label)
        return
    U.safe(fn, **kwargs)

## 1. SageMaker AI
Endpoints first - they are the main running cost.

In [ ]:
sm = U.client("sagemaker")
endpoints = [e["EndpointName"] for e in sm.list_endpoints(NameContains="safedrive", MaxResults=100)["Endpoints"]]
print(endpoints)

Monitoring schedules must go before their endpoint.

In [ ]:
schedules = sm.list_monitoring_schedules(NameContains="safedrive")["MonitoringScheduleSummaries"]
for s in schedules:
    drop(sm.delete_monitoring_schedule, MonitoringScheduleName=s["MonitoringScheduleName"])

Inference components must go before their endpoint.

In [ ]:
components = sm.list_inference_components(NameContains="safedrive")["InferenceComponents"]
for c in components:
    drop(sm.delete_inference_component, InferenceComponentName=c["InferenceComponentName"])

In [ ]:
for name in endpoints:
    drop(sm.delete_endpoint, EndpointName=name)

Endpoint configurations and models (free, but tidy).

In [ ]:
configs = sm.list_endpoint_configs(NameContains="safedrive", MaxResults=100)["EndpointConfigs"]
for c in configs:
    drop(sm.delete_endpoint_config, EndpointConfigName=c["EndpointConfigName"])

In [ ]:
models = sm.list_models(NameContains="safedrive", MaxResults=100)["Models"]
for m in models:
    drop(sm.delete_model, ModelName=m["ModelName"])

Pipeline, then the model package group (its versions must be deleted first).

In [ ]:
drop(sm.delete_pipeline, PipelineName="safedrive-fraud-pipeline")

In [ ]:
try:
    versions = sm.list_model_packages(ModelPackageGroupName="safedrive-fraud")["ModelPackageSummaryList"]
except Exception as e:
    versions = []
    print("no model package group:", str(e)[:80])
for v in versions:
    drop(sm.delete_model_package, ModelPackageName=v["ModelPackageArn"])

In [ ]:
drop(sm.delete_model_package_group, ModelPackageGroupName="safedrive-fraud")

Feature group (online and offline store) and the MLflow tracking server (hourly cost if left running).

In [ ]:
drop(sm.delete_feature_group, FeatureGroupName="safedrive-claims-fg")

In [ ]:
drop(sm.delete_mlflow_tracking_server, TrackingServerName="safedrive-mlflow")

## 2. Bedrock agents and AgentCore

In [ ]:
agent = U.client("bedrock-agent")
agents = [a for a in agent.list_agents(maxResults=100)["agentSummaries"] if mine(a["agentName"])]
for a in agents:
    drop(agent.delete_agent, agentId=a["agentId"], skipResourceInUseCheck=True)

AgentCore: runtimes, gateways (targets first) and memories.

In [ ]:
acc = U.client("bedrock-agentcore-control")
runtimes = [r for r in acc.list_agent_runtimes(maxResults=100)["agentRuntimes"] if mine(r["agentRuntimeName"])]
for r in runtimes:
    drop(acc.delete_agent_runtime, agentRuntimeId=r["agentRuntimeId"])

In [ ]:
gateways = [g for g in acc.list_gateways(maxResults=100)["items"] if mine(g["name"])]
for g in gateways:
    for t in acc.list_gateway_targets(gatewayIdentifier=g["gatewayId"])["items"]:
        drop(acc.delete_gateway_target, gatewayIdentifier=g["gatewayId"], targetId=t["targetId"])
    drop(acc.delete_gateway, gatewayIdentifier=g["gatewayId"])

In [ ]:
memories = [m for m in acc.list_memories(maxResults=100)["memories"] if mine(m["id"])]
for m in memories:
    drop(acc.delete_memory, memoryId=m["id"])

## 3. Knowledge Base and S3 Vectors
Delete the data sources, then the Knowledge Base, then the vector indexes and the vector bucket.

In [ ]:
kbs = [k for k in agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"] if mine(k["name"])]
for k in kbs:
    for d in agent.list_data_sources(knowledgeBaseId=k["knowledgeBaseId"])["dataSourceSummaries"]:
        drop(agent.delete_data_source, knowledgeBaseId=k["knowledgeBaseId"], dataSourceId=d["dataSourceId"])
    drop(agent.delete_knowledge_base, knowledgeBaseId=k["knowledgeBaseId"])

In [ ]:
s3v = U.client("s3vectors")
VBUCKET = f"safedrive-vectors-{U.account_id()}"
try:
    indexes = s3v.list_indexes(vectorBucketName=VBUCKET)["indexes"]
except Exception as e:
    indexes = []
    print("no vector bucket:", str(e)[:80])
for i in indexes:
    drop(s3v.delete_index, vectorBucketName=VBUCKET, indexName=i["indexName"])

In [ ]:
drop(s3v.delete_vector_bucket, vectorBucketName=VBUCKET)

OpenSearch Serverless collections (notebook 1.1b). These bill by the hour, so make sure none is left.

In [ ]:
aoss = U.client("opensearchserverless")
for c in [c for c in aoss.list_collections()["collectionSummaries"] if mine(c["name"])]:
    drop(aoss.delete_collection, id=c["id"])

Their access and security policies.

In [ ]:
for p in aoss.list_access_policies(type="data")["accessPolicySummaries"]:
    if mine(p["name"]):
        drop(aoss.delete_access_policy, name=p["name"], type="data")
for kind in ["network", "encryption"]:
    for p in aoss.list_security_policies(type=kind)["securityPolicySummaries"]:
        if mine(p["name"]):
            drop(aoss.delete_security_policy, name=p["name"], type=kind)

## 4. Other Bedrock resources
Guardrails, prompts and application inference profiles.

In [ ]:
bedrock = U.client("bedrock")
guardrails = [g for g in bedrock.list_guardrails(maxResults=100)["guardrails"] if mine(g["name"])]
for g in guardrails:
    drop(bedrock.delete_guardrail, guardrailIdentifier=g["id"])

In [ ]:
prompts = [p for p in agent.list_prompts(maxResults=100)["promptSummaries"] if mine(p["name"])]
for p in prompts:
    drop(agent.delete_prompt, promptIdentifier=p["id"])

In [ ]:
profiles = bedrock.list_inference_profiles(typeEquals="APPLICATION")["inferenceProfileSummaries"]
for p in [p for p in profiles if mine(p["inferenceProfileName"])]:
    drop(bedrock.delete_inference_profile, inferenceProfileIdentifier=p["inferenceProfileArn"])

Custom models exist only if you ran the 🔒 fine-tuning cells.

In [ ]:
custom = [m for m in bedrock.list_custom_models()["modelSummaries"] if mine(m["modelName"])]
for m in custom:
    drop(bedrock.delete_custom_model, modelIdentifier=m["modelArn"])

> **Model invocation logging** (notebook 4.1b) is an account-level setting. If you turned it on only for this
> practice, turn it off with `bedrock.delete_model_invocation_logging_configuration()`.

## 5. Lambda, EventBridge, Scheduler, SNS

In [ ]:
lam = U.client("lambda")
functions = [f["FunctionName"] for f in lam.list_functions()["Functions"] if mine(f["FunctionName"])]
for name in functions:
    drop(lam.delete_function, FunctionName=name)

A rule can only be deleted after its targets are removed.

In [ ]:
events = U.client("events")
rules = [r["Name"] for r in events.list_rules(NamePrefix="safedrive")["Rules"]]
for name in rules:
    ids = [t["Id"] for t in events.list_targets_by_rule(Rule=name)["Targets"]]
    if ids and CONFIRM:
        U.safe(events.remove_targets, Rule=name, Ids=ids)
    drop(events.delete_rule, Name=name)

In [ ]:
scheduler = U.client("scheduler")
for s in scheduler.list_schedules(NamePrefix="safedrive")["Schedules"]:
    drop(scheduler.delete_schedule, Name=s["Name"])

In [ ]:
sns = U.client("sns")
topics = [t["TopicArn"] for t in sns.list_topics()["Topics"] if mine(t["TopicArn"].split(":")[-1])]
for arn in topics:
    drop(sns.delete_topic, TopicArn=arn)

## 6. CloudWatch: alarms, dashboards, log groups

In [ ]:
cw = U.client("cloudwatch")
alarms = [a["AlarmName"] for a in cw.describe_alarms(AlarmNamePrefix="safedrive")["MetricAlarms"]]
print(alarms)
if alarms:
    drop(cw.delete_alarms, AlarmNames=alarms)

In [ ]:
dashboards = [d["DashboardName"] for d in cw.list_dashboards(DashboardNamePrefix="safedrive")["DashboardEntries"]]
if dashboards:
    drop(cw.delete_dashboards, DashboardNames=dashboards)

Log groups whose name contains `safedrive` (Lambda, endpoints, CodeBuild, Bedrock logging).

In [ ]:
logs = U.client("logs")
groups = logs.describe_log_groups(logGroupNamePattern="safedrive", limit=50)["logGroups"]
for g in groups:
    drop(logs.delete_log_group, logGroupName=g["logGroupName"])

## 7. Data services: DynamoDB, Glue, Kinesis, Firehose

In [ ]:
ddb = U.client("dynamodb")
drop(ddb.delete_table, TableName=C.CLAIMS_TABLE)

Deleting a Glue database also deletes its tables (metadata only; the S3 data goes with the bucket).

In [ ]:
glue = U.client("glue")
drop(glue.delete_database, Name="safedrive")

In [ ]:
for name in [j for j in glue.list_jobs()["JobNames"] if mine(j)]:
    drop(glue.delete_job, JobName=name)

In [ ]:
for name in [c for c in glue.list_crawlers()["CrawlerNames"] if mine(c)]:
    drop(glue.delete_crawler, Name=name)

In [ ]:
kinesis = U.client("kinesis")
for name in [s for s in kinesis.list_streams()["StreamNames"] if mine(s)]:
    drop(kinesis.delete_stream, StreamName=name, EnforceConsumerDeletion=True)

In [ ]:
firehose = U.client("firehose")
for name in [s for s in firehose.list_delivery_streams()["DeliveryStreamNames"] if mine(s)]:
    drop(firehose.delete_delivery_stream, DeliveryStreamName=name)

## 8. CI/CD and infrastructure: CodePipeline, CodeBuild, ECR, CloudFormation, VPC endpoints

In [ ]:
codepipeline = U.client("codepipeline")
for p in [p["name"] for p in codepipeline.list_pipelines()["pipelines"] if mine(p["name"])]:
    drop(codepipeline.delete_pipeline, name=p)

In [ ]:
codebuild = U.client("codebuild")
for p in [p for p in codebuild.list_projects()["projects"] if mine(p)]:
    drop(codebuild.delete_project, name=p)

In [ ]:
ecr = U.client("ecr")
repos = [r["repositoryName"] for r in ecr.describe_repositories()["repositories"] if mine(r["repositoryName"])]
for r in repos:
    drop(ecr.delete_repository, repositoryName=r, force=True)

CloudFormation stacks. A stack that **imports** values must be deleted before the stack that exports
them, so run this cell twice (a minute apart) if the first pass reports an export in use.

In [ ]:
cfn = U.client("cloudformation")
live = ["CREATE_COMPLETE", "UPDATE_COMPLETE", "ROLLBACK_COMPLETE", "DELETE_FAILED", "UPDATE_ROLLBACK_COMPLETE"]
stacks = [s["StackName"] for s in cfn.list_stacks(StackStatusFilter=live)["StackSummaries"] if mine(s["StackName"])]
for name in sorted(stacks, reverse=True):
    drop(cfn.delete_stack, StackName=name)

VPC endpoints created outside CloudFormation (notebook 3.2a), found by the project tag.

In [ ]:
ec2 = U.client("ec2")
tag_filter = [{"Name": "tag:project", "Values": ["mla-c02-practice"]}]
vpce = [v["VpcEndpointId"] for v in ec2.describe_vpc_endpoints(Filters=tag_filter)["VpcEndpoints"]]
print(vpce)
if vpce:
    drop(ec2.delete_vpc_endpoints, VpcEndpointIds=vpce)

## 9. S3 buckets
This deletes **all practice data, models and the shared state file**. The helper also removes old
object versions, which a versioned bucket needs before it can be deleted.

In [ ]:
def empty_and_delete(bucket_name):
    if not CONFIRM:
        print("would empty and delete bucket:", bucket_name)
        return
    bucket = U.session.resource("s3").Bucket(bucket_name)
    U.safe(bucket.object_versions.delete)
    U.safe(bucket.objects.all().delete)
    U.safe(bucket.delete)

The CI/CD artifact bucket from notebook 3.3a (if it is still there).

In [ ]:
empty_and_delete(f"safedrive-cicd-{U.account_id()}-{U.REGION}")

The KMS demo bucket from notebook 4.3a (if it is still there).

In [ ]:
empty_and_delete(f"safedrive-secure-{U.account_id()}-{U.REGION}")

The main practice bucket.

In [ ]:
empty_and_delete(U.bucket())

## 10. IAM roles
Last of all: the extra roles from notebook 4.3a and the practice role. Inline policies are removed
and managed policies detached before a role can be deleted.

In [ ]:
iam = U.client("iam")

def delete_role(role_name):
    if not CONFIRM:
        print("would delete role:", role_name)
        return
    try:
        for p in iam.list_role_policies(RoleName=role_name)["PolicyNames"]:
            iam.delete_role_policy(RoleName=role_name, PolicyName=p)
        for p in iam.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]:
            iam.detach_role_policy(RoleName=role_name, PolicyArn=p["PolicyArn"])
        iam.delete_role(RoleName=role_name)
        print("deleted role:", role_name)
    except Exception as e:
        print("skipped:", role_name, "-", str(e)[:120])

In [ ]:
roles = [r["RoleName"] for r in iam.list_roles(MaxItems=1000)["Roles"] if mine(r["RoleName"])]
print(roles)

In [ ]:
for name in roles:
    delete_role(name)

Customer-managed IAM policies named `safedrive*` (if any were created).

In [ ]:
policies = [p for p in iam.list_policies(Scope="Local")["Policies"] if mine(p["PolicyName"])]
for p in policies:
    drop(iam.delete_policy, PolicyArn=p["Arn"])

## 11. Final check
Anything still carrying the project tag? (Recently deleted resources can linger here for a while.)

In [ ]:
tagging = U.client("resourcegroupstaggingapi")
left = tagging.get_resources(TagFilters=[{"Key": "project", "Values": ["mla-c02-practice"]}])
for r in left["ResourceTagMappingList"]:
    print(r["ResourceARN"])
print(len(left["ResourceTagMappingList"]), "tagged resources still listed")

**Also check by hand**
- KMS key `alias/safedrive` (notebook 4.3a): keys cannot be deleted at once, only scheduled for deletion (7-30 days).
- AWS Budgets and Config rules named `safedrive*`, if you created them in 4.2 / 4.3a.
- The local `data/` folder and `scripts/` folders in this repository.